# Optuna AdamW Hyperparameter Search

Search AdamW-centered hyperparameters for two workloads:

- **Protein pretrain** from `config/train.yaml`
- **Protein completion** / profile-aware span completion from `config/profile_span_completion.yaml`

The search follows the small-LLM training pattern from `rasbt/LLMs-from-scratch`: AdamW, validation loss as the objective, short proxy runs, gradient clipping, and a compact search space around learning rate, weight decay, batch shape, context length, gradient accumulation, and evaluation cadence. Run short trials first, then copy the best YAML into a longer training run.

In [1]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import shutil
from pathlib import Path
from typing import Any

import optuna
import torch
import yaml

from libs.notebook_runtime import bootstrap_notebook
from libs.core.pretrain.protein_lm.trainer import ProteinPretrainTrainer
from libs.instruction.trainer import InstructionTrainer, build_instruction_training_config
from libs.notebook_runtime import apply_instruction_notebook_overrides

RUNTIME = bootstrap_notebook(globals().get("__file__", Path.cwd()))
REPO_DIR = Path(RUNTIME["repo_dir"]).resolve()
print(json.dumps(RUNTIME, indent=2))

c:\Users\Admin\Desktop\MDNAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "repo_dir": "C:\\Users\\Admin\\Desktop\\MDNAC",
  "platform": "Windows",
  "platform_name": "Windows",
  "is_colab": false,
  "is_notebook": true,
  "python": "3.11.15",
  "cuda_available": true,
  "cuda_device_count": 1
}


## Search Settings

Use small proxy runs for Optuna. Increase `N_TRIALS_*` and `*_TRIAL_MAX_STEPS` after the pipeline is confirmed on your machine.

In [2]:
# Base configs
PRETRAIN_BASE_CONFIG = Path(os.environ.get("MDNAC_OPTUNA_PRETRAIN_CONFIG", REPO_DIR / "config" / "train.yaml")).expanduser()
COMPLETION_BASE_CONFIG = Path(os.environ.get("MDNAC_OPTUNA_COMPLETION_CONFIG", REPO_DIR / "config" / "profile_span_completion.yaml")).expanduser()

# Study outputs
STUDY_ROOT = Path(os.environ.get("MDNAC_OPTUNA_DIR", REPO_DIR / "data" / "optuna" / "adamw_search")).expanduser().resolve()
STORAGE_URL = f"sqlite:///{(STUDY_ROOT / 'optuna_studies.db').as_posix()}"

# Trial budgets. Keep these short for search, then retrain with the winning YAML.
N_TRIALS_PRETRAIN = int(os.environ.get("MDNAC_OPTUNA_PRETRAIN_TRIALS", "12"))
N_TRIALS_COMPLETION = int(os.environ.get("MDNAC_OPTUNA_COMPLETION_TRIALS", "12"))
PRETRAIN_TRIAL_MAX_STEPS = int(os.environ.get("MDNAC_OPTUNA_PRETRAIN_MAX_STEPS", "150"))
COMPLETION_TRIAL_MAX_STEPS = int(os.environ.get("MDNAC_OPTUNA_COMPLETION_MAX_STEPS", "150"))
EVAL_BATCHES = int(os.environ.get("MDNAC_OPTUNA_EVAL_BATCHES", "8"))
SEED = int(os.environ.get("MDNAC_OPTUNA_SEED", "123"))

STUDY_ROOT.mkdir(parents=True, exist_ok=True)
print("study_root:", STUDY_ROOT)
print("storage:", STORAGE_URL)

study_root: C:\Users\Admin\Desktop\MDNAC\data\optuna\adamw_search
storage: sqlite:///C:/Users/Admin/Desktop/MDNAC/data/optuna/adamw_search/optuna_studies.db


## Helpers

In [3]:
def load_yaml(path: Path) -> dict[str, Any]:
    resolved = path if path.is_absolute() else (REPO_DIR / path).resolve()
    data = yaml.safe_load(resolved.read_text(encoding="utf-8")) or {}
    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML mapping: {resolved}")
    return data


def deep_update(base: dict[str, Any], updates: dict[str, Any]) -> dict[str, Any]:
    result = copy.deepcopy(base)
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update(result[key], value)
        else:
            result[key] = value
    return result


def dump_trial_yaml(config: dict[str, Any], path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return path


def cleanup_torch() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def best_loss_from_result(result) -> float:
    candidates = [getattr(result, "best_loss", None), getattr(result, "best_val_loss", None), getattr(result, "final_val_loss", None)]
    for value in candidates:
        if value is not None and math.isfinite(float(value)):
            return float(value)
    raise optuna.TrialPruned("Trial did not produce a finite validation loss.")


def suggest_adamw_common(trial: optuna.Trial, *, prefix: str = "") -> dict[str, Any]:
    return {
        f"{prefix}learning_rate": trial.suggest_float(f"{prefix}learning_rate", 1e-5, 8e-4, log=True),
        f"{prefix}weight_decay": trial.suggest_float(f"{prefix}weight_decay", 0.0, 0.2),
        f"{prefix}gradient_accumulation_steps": trial.suggest_categorical(
            f"{prefix}gradient_accumulation_steps", [1, 2, 4, 8, 16]
        ),
    }


def write_best_summary(study: optuna.Study, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "study_name": study.study_name,
        "best_value": study.best_value,
        "best_params": study.best_params,
        "best_trial_number": study.best_trial.number,
        "datetime_start": str(study.best_trial.datetime_start),
        "datetime_complete": str(study.best_trial.datetime_complete),
    }
    (output_dir / "best_trial.json").write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(json.dumps(payload, indent=2, ensure_ascii=False))

## Protein Pretrain Objective

In [4]:
def build_pretrain_trial_config(trial: optuna.Trial) -> tuple[dict[str, Any], Path]:
    base = load_yaml(PRETRAIN_BASE_CONFIG)
    params = suggest_adamw_common(trial)
    batch_size = trial.suggest_categorical("batch_size", [1, 2, 4, 8])
    context_length = trial.suggest_categorical("context_length", [256, 384, 512, 768, 1024])
    eval_freq = trial.suggest_categorical("eval_freq", [25, 50, 100])

    model_updates = {"context_length": context_length, "stride": max(1, context_length // 2)}

    trial_dir = STUDY_ROOT / "pretrain" / f"trial_{trial.number:04d}"
    config = deep_update(
        base,
        {
            "mode": {"name": "train_from_scratch", "resume_if_available": False},
            "data": {"batch_size": batch_size, "num_workers": 0, "pin_memory": "auto", "shuffle_examples": True},
            "model": model_updates,
            "training": {
                "max_steps": PRETRAIN_TRIAL_MAX_STEPS,
                "num_epochs": 1,
                "gradient_accumulation_steps": params["gradient_accumulation_steps"],
                "eval_freq": eval_freq,
                "eval_batches": EVAL_BATCHES,
                "save_every_steps": None,
                "save_last": False,
                "save_best": False,
                "save_final": False,
                "grad_clip_norm": 1.0,
            },
            "optimizer": {
                "type": "adamw",
                "learning_rate": params["learning_rate"],
                "weight_decay": params["weight_decay"],
                "fused": "auto",
            },
            "runtime": {"multi_gpu_mode": "none", "preflight_vram_check": False, "seed": SEED},
            "resume": {
                "restore_optimizer_state": False,
                "checkpoint_path": str(trial_dir / "checkpoint_last.pt"),
                "output_checkpoint_path": str(trial_dir / "checkpoint_last.pt"),
                "best_checkpoint_path": str(trial_dir / "checkpoint_best.pt"),
                "final_checkpoint_path": str(trial_dir / "checkpoint_final.pt"),
                "resume_state_path": str(trial_dir / "resume_state.json"),
            },
            "paths": {
                "checkpoint_dir": str(trial_dir),
                "resume_state_path": str(trial_dir / "resume_state.json"),
                "metrics_history_path": str(trial_dir / "metrics_history.jsonl"),
                "training_config_snapshot_path": str(trial_dir / "training_config.snapshot.json"),
            },
        },
    )
    return config, trial_dir


def pretrain_objective(trial: optuna.Trial) -> float:
    torch.manual_seed(SEED + trial.number)
    config, trial_dir = build_pretrain_trial_config(trial)
    config_path = dump_trial_yaml(config, trial_dir / "trial_config.yaml")
    try:
        trainer = ProteinPretrainTrainer(REPO_DIR, config_path=config_path)
        result = trainer.train()
        objective = best_loss_from_result(result)
        trial.set_user_attr("trial_dir", str(trial_dir))
        trial.set_user_attr("config_path", str(config_path))
        trial.set_user_attr("global_step", int(result.global_step))
        trial.set_user_attr("tokens_seen", int(result.tokens_seen))
        return objective
    except torch.cuda.OutOfMemoryError:
        raise optuna.TrialPruned("CUDA OOM")
    finally:
        cleanup_torch()

In [ ]:
pretrain_study = optuna.create_study(
    study_name="mdnac_adamw_pretrain",
    direction="minimize",
    storage=STORAGE_URL,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
)
pretrain_study.optimize(pretrain_objective, n_trials=N_TRIALS_PRETRAIN, gc_after_trial=True)
write_best_summary(pretrain_study, STUDY_ROOT / "pretrain")

[I 2026-06-08 16:38:10,062] Using an existing study with name 'mdnac_adamw_pretrain' instead of creating a new one.


🚀 Training mode: train_from_scratch
📦 [1/6] Loading tokenizer...
✅ Tokenizer loaded — vocab_size=256
🧠 [2/6] Building model...
The MDC fast path is unavailable (missing optional libraries: causal-conv1d, flash-linear-attention). Falling back to the torch implementation. The fallback uses fp32 computation (2x VRAM for activations).
✅ Model built — 280,021,632 parameters
⚙️  [3/6] Preparing runtime...
✅ Device: cuda | Distributed: False
🔧 [4/6] Creating optimizer...
✅ Optimizer ready
📂 [5/6] Building data loaders...
✅ Data loaders ready
🏋️ [6/6] Starting training loop...
📈 Epoch 1/1 | step=0/150 | tokens=0
  🔄 step=50 | loss=4.5930 | tokens=151,414


## Protein Completion Objective

In [ ]:
def build_completion_trial_config(trial: optuna.Trial) -> tuple[dict[str, Any], Path]:
    base = load_yaml(COMPLETION_BASE_CONFIG)
    params = suggest_adamw_common(trial, prefix="completion_")
    batch_size = trial.suggest_categorical("completion_batch_size", [1, 2, 4, 8])
    eval_freq = trial.suggest_categorical("completion_eval_freq", [25, 50, 100])

    trial_dir = STUDY_ROOT / "protein_completion" / f"trial_{trial.number:04d}"
    config = deep_update(
        base,
        {
            "data": {"batch_size": batch_size, "num_workers": 0, "pin_memory": "auto"},
            "training": {
                "max_steps": COMPLETION_TRIAL_MAX_STEPS,
                "num_epochs": 1,
                "gradient_accumulation_steps": params["completion_gradient_accumulation_steps"],
                "eval_freq": eval_freq,
                "eval_batches": EVAL_BATCHES,
                "save_every_steps": None,
                "save_last": False,
                "save_best": False,
                "save_final": False,
                "grad_clip_norm": 1.0,
            },
            "optimizer": {
                "type": "adamw",
                "learning_rate": params["completion_learning_rate"],
                "weight_decay": params["completion_weight_decay"],
                "fused": "auto",
            },
            "runtime": {"multi_gpu_mode": "none", "seed": SEED + trial.number},
            "resume": {"resume_if_available": False, "restore_optimizer_state": False},
            "paths": {"checkpoint_dir": str(trial_dir), "output_dir": str(trial_dir)},
        },
    )
    return config, trial_dir


def completion_objective(trial: optuna.Trial) -> float:
    torch.manual_seed(SEED + trial.number)
    config, trial_dir = build_completion_trial_config(trial)
    config_path = dump_trial_yaml(config, trial_dir / "trial_config.yaml")
    try:
        training_config = build_instruction_training_config(REPO_DIR, config)
        training_config = apply_instruction_notebook_overrides(training_config)
        trainer = InstructionTrainer(training_config)
        result = trainer.train()
        objective = best_loss_from_result(result)
        trial.set_user_attr("trial_dir", str(trial_dir))
        trial.set_user_attr("config_path", str(config_path))
        trial.set_user_attr("global_step", int(result.global_step))
        trial.set_user_attr("tokens_seen", int(result.tokens_seen))
        return objective
    except torch.cuda.OutOfMemoryError:
        raise optuna.TrialPruned("CUDA OOM")
    finally:
        cleanup_torch()

In [ ]:
completion_study = optuna.create_study(
    study_name="mdnac_adamw_protein_completion",
    direction="minimize",
    storage=STORAGE_URL,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
)
completion_study.optimize(completion_objective, n_trials=N_TRIALS_COMPLETION, gc_after_trial=True)
write_best_summary(completion_study, STUDY_ROOT / "protein_completion")

## Export Best YAMLs

These cells copy the best trial YAMLs to stable files you can use for real training runs.

In [ ]:
def export_best_config(study: optuna.Study, target_path: Path) -> Path:
    config_path = Path(study.best_trial.user_attrs["config_path"])
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(config_path, target_path)
    return target_path

pretrain_best_yaml = export_best_config(pretrain_study, STUDY_ROOT / "best_pretrain_adamw.yaml")
completion_best_yaml = export_best_config(completion_study, STUDY_ROOT / "best_protein_completion_adamw.yaml")
print("pretrain_best_yaml:", pretrain_best_yaml)
print("completion_best_yaml:", completion_best_yaml)